# 03 · Moral agents: full fine-tuning (arm A) vs LoRA (arm B)

Runtime > Change runtime type > **T4 GPU**.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

Mounted at /content/drive
Fri Nov 20 19:24:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

In [10]:
import getpass, os
GITHUB_USER = 'natitedros'
REPO = 'CMVA-context-based-aggregation'
token = getpass.getpass('GitHub token (leave blank if repo is public): ')
auth = f'{token}@' if token else ''
%cd /content
!rm -rf /content/repo
!git clone -q https://{auth}github.com/{GITHUB_USER}/{REPO}.git /content/repo
%cd /content/repo
del token, auth

GitHub token (leave blank if repo is public): ··········
/content
/content/repo


In [8]:
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao

In [4]:
import os
os.environ['PROJECT_ROOT'] = '/content/drive/MyDrive/NLP-Final'
os.environ['SCRATCH_DIR'] = '/content/scratch'

### 1. Build the training data (once)
One file per value plus the control. Check how many qualifying pairs each value has.

In [5]:
!python src/prepare_agent_data.py

Loaded 113,828 rows. Columns: ['chatbot', 'Q', 'A', 'rot', 'moral', 'moral-vector', 'A_agrees', 'rot-agree', 'violation-severity', 'worker_answer', 'comments', 'QA', 'split']
Dropping 2 malformed rows (non-numeric A_agrees).
A_agrees values: {2: 66261, 0: 23166, 1: 18766}
split values: {'train': 86529, 'dev': 10842, 'test': 10817}
Sample 'moral' values: ['loyalty', 'care|fairness', 'care', 'authority|sanctity', 'authority']
Ignoring 5 rows with an unrecognised split value.
Using MIC's own train/dev/test split.
86,453 train-split annotations with answers of >= 3 words
care       14,758 qualifying pairs -> 2,000 used
fairness    7,319 qualifying pairs -> 2,000 used
loyalty     7,358 qualifying pairs -> 2,000 used
authority   6,412 qualifying pairs -> 2,000 used
sanctity    4,079 qualifying pairs -> 2,000 used
control    30,212 unique pairs -> 2,000 used
Saved to /content/drive/MyDrive/NLP-Final/data/agent_data


### 2. Smoke test both arms on 64 examples
The full fine-tune is the one that could run out of memory, so this checks it in about a minute.

In [11]:
!python src/train_agent.py --mode lora --value care --n_train 64 --epochs 1 --tag _smoke --overwrite
!python src/train_agent.py --mode full --value care --n_train 64 --epochs 1 --tag _smoke --overwrite

64 training examples for lora_care_smoke
Loading weights: 100% 290/290 [00:00<00:00, 386.43it/s]
Trainable parameters: 8,798,208 of 502,830,976 (1.75%)
{'train_runtime': '5.235', 'train_samples_per_second': '12.22', 'train_steps_per_second': '0.764', 'train_loss': '2.405', 'epoch': '1'}
100% 4/4 [00:05<00:00,  1.31s/it]
{
  "agent": "lora_care_smoke",
  "mode": "lora",
  "value": "care",
  "rank": 16,
  "lr": 0.0002,
  "epochs": 1,
  "effective_batch": 16,
  "train_examples": 64,
  "trainable_params": 8798208,
  "total_params": 502830976,
  "trainable_pct": 1.75,
  "train_seconds": 5.7,
  "peak_train_gpu_mb": 5983,
  "first_loss": null,
  "final_loss": null,
  "avg_train_loss": 2.405,
  "saved_size_mb": 33.6,
  "grad_ckpt": false
}
Saved agent to /content/drive/MyDrive/NLP-Final/models/agents/lora_care_smoke
64 training examples for full_care_smoke
Loading weights: 100% 290/290 [00:00<00:00, 426.30it/s]
Trainable parameters: 494,032,768 of 494,032,768 (100.00%)
{'train_runtime': '3.768

In [12]:
# Remove the smoke-test models so they don't use Drive space
!rm -rf "$PROJECT_ROOT"/models/agents/*_smoke "$PROJECT_ROOT"/results/agent_*_smoke.json

### 3. Train the care agent in both arms

In [13]:
!python src/train_agent.py --mode lora --value care
!python src/train_agent.py --mode full --value care

2000 training examples for lora_care
Loading weights: 100% 290/290 [00:00<00:00, 382.02it/s]
Trainable parameters: 8,798,208 of 502,830,976 (1.75%)
{'loss': '2.199', 'grad_norm': '1.926', 'learning_rate': '0.0001992', 'epoch': '0.08'}
{'loss': '1.922', 'grad_norm': '1.791', 'learning_rate': '0.0001909', 'epoch': '0.16'}
{'loss': '1.844', 'grad_norm': '1.678', 'learning_rate': '0.0001826', 'epoch': '0.24'}
{'loss': '1.875', 'grad_norm': '1.736', 'learning_rate': '0.0001744', 'epoch': '0.32'}
{'loss': '1.938', 'grad_norm': '1.511', 'learning_rate': '0.0001661', 'epoch': '0.4'}
{'loss': '1.815', 'grad_norm': '1.943', 'learning_rate': '0.0001579', 'epoch': '0.48'}
{'loss': '1.852', 'grad_norm': '1.347', 'learning_rate': '0.0001496', 'epoch': '0.56'}
{'loss': '1.804', 'grad_norm': '1.954', 'learning_rate': '0.0001413', 'epoch': '0.64'}
{'loss': '1.775', 'grad_norm': '1.502', 'learning_rate': '0.0001331', 'epoch': '0.72'}
{'loss': '1.713', 'grad_norm': '1.615', 'learning_rate': '0.0001248', 

### 4. Score both care agents on the 500 held-out questions

In [14]:
!python src/generate_and_score.py --tag lora_care --adapter "$PROJECT_ROOT/models/agents/lora_care" --n 500
!python src/generate_and_score.py --tag full_care --model "$PROJECT_ROOT/models/agents/full_care" --n 500

Generating 500 answers with /content/drive/MyDrive/NLP-Final/models/agents/lora_care
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:00<00:00, 373.12it/s]
  generated 16/500
  generated 32/500
  generated 48/500
  generated 64/500
  generated 80/500
  generated 96/500
  generated 112/500
  generated 128/500
  generated 144/500
  generated 160/500
  generated 176/500
  generated 192/500
  generated 208/500
  generated 224/500
  generated 240/500
  generated 256/500
  generated 272/500
  generated 288/500
  generated 304/500
  generated 320/500
  generated 336/500
  generated 352/500
  generated 368/500
  generated 384/500
  generated 400/500
  generated 416/500
  generated 432/500
  generated 448/500
  generated 464/500
  generated 480/500
  generated 496/500
  generated 500/500
Loading weights: 100% 201/201 [00:10<00:00, 20.04it/s]
care       expected reward 0.733
Loading weights: 100% 201/201 [00:06<00:00, 31.87it/s] 
fairness   expec

In [15]:
import pandas as pd
pd.set_option('display.max_colwidth', 300)
R = os.environ['PROJECT_ROOT'] + '/results'
for tag in ['pt', 'lora_care', 'full_care']:
    df = pd.read_csv(f'{R}/generations_{tag}.csv')
    print(tag, '\n', df['reply'].head(3).tolist(), '\n')

pt 
 ["The widespread reverence for the ground soldiers who walk on the streets of cities and towns, despite their often brutal nature, can be attributed to several factors:\n\n1. **Historical Context**: The military has been an integral part of many societies throughout history, serving as both a means of defense and a symbol of national pride. Soldiers have often faced immense danger and sacrifice in their roles.\n\n2. **Symbolic Significance**: In some cultures, the ground soldiers' actions are seen as embodying the very essence of the nation's identity and values. They are often depicted as heroes or symbols of strength and resilience.\n\n3. **Cultural Norm", "As an AI language model, I don't have personal preferences or emotions like humans do. However, I can tell you that my creators and developers strive to create content that is respectful, accurate, and aligned with the values of their home country. They work hard to ensure that any information they provide is free from bias a